# Probability Flow ODE vs SDE

Chapter 20: ODE, SDE, and Continuous Limits of Algorithms · *Mathematics for AI and Machine Learning* by Fuheng Wu

One deterministic and one stochastic evolution under shared drift parameters.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-20/pf-ode-vs-sde/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch

torch.manual_seed(42)

# The demo's defaults: dX = -lam X dt + sig dW from X0 = x0, T = 4, dt = 0.02
lam, sig, x0, T, dt = 0.9, 0.45, 2.2, 4.0, 0.02
n = round(T / dt)

# Exact marginals from a point start: N(x0 e^{-lam t}, sig^2 (1 - e^{-2 lam t}) / (2 lam))
mean_T = x0 * math.exp(-lam * T)
var_T = sig ** 2 * (1 - math.exp(-2 * lam * T)) / (2 * lam)

# ── Many SDE paths ──────────────────────────────────────────────────────────
x = torch.full((20000,), x0, dtype=torch.float64)
for _ in range(n):
    x = x - lam * x * dt + sig * math.sqrt(dt) * torch.randn_like(x)
print(f"SDE at T: mean {x.mean().item():.4f} (exact {mean_T:.4f}), var {x.var().item():.4f} (exact {var_T:.4f})")

# ── Probability-flow ODE: dx/dt = -lam x - (sig^2/2) d/dx log p_t(x) ────────
# Start it from samples of p_s at a small time s (a point mass has no density yet).
s = 0.05
m = lambda t: x0 * math.exp(-lam * t)
v = lambda t: sig ** 2 * (1 - math.exp(-2 * lam * t)) / (2 * lam)
y = m(s) + math.sqrt(v(s)) * torch.randn(20000, dtype=torch.float64)
t = s
h = (T - s) / 2000
for _ in range(2000):
    score = -(y - m(t)) / v(t)
    y = y + (-lam * y - 0.5 * sig ** 2 * score) * h       # no noise: each point moves deterministically
    t += h
print(f"PF-ODE at T: mean {y.mean().item():.4f}, var {y.var().item():.4f}  -> same distribution as the SDE")

# From the single point x0 the PF-ODE path is the mean curve (what the demo's blue line shows)
print(f"one deterministic path from x0 ends at {mean_T:.4f}; an SDE path ends anywhere in that spread")

SDE at T: mean 0.0551 (exact 0.0601), var 0.1149 (exact 0.1124)
PF-ODE at T: mean 0.0603, var 0.1149  -> same distribution as the SDE
one deterministic path from x0 ends at 0.0601; an SDE path ends anywhere in that spread
